# FGI Data Import
Builds `FGI_liveState.xlsx` from the raw import workbooks. 


In [58]:
## LIBRARY IMPORTS
from pathlib import Path
import numpy as np
import pandas as pd


## 1. Local Paths
Resolve the repository from the current working directory and define raw workbook inputs plus the generated live-state output.


In [59]:
# FILEPATHS FOR DATA IMPORT
rootpath = Path.cwd()
if not (rootpath / 'input').exists():
    raise FileNotFoundError(f'Input directory not found under {rootpath}')

filepaths = {
    'FAstatus': rootpath / 'input' / 'FA_Status_FGI_Handoff.xlsx',
    'FGI_Locations': rootpath / 'input' / 'FGI_Locations_wPriority.xlsx',
    'Nodes': rootpath / 'input' / 'Nodes.xlsx',
    'Move Times': rootpath / 'input' / 'move_time_estimation.xlsx',
    'Paint Schedule': rootpath / 'input' / 'paint_schedules.xlsx'
}

LIVE_STATE_FILE = rootpath / 'FGI_liveState.xlsx'


## 2. Data Import Defaults
Defines labor defaults, CPJ values, centerline dependencies, and BTG-to-team mappings exported into live state or used to shape imported data.


In [60]:
## DATA IMPORT DEFAULTS

FGI_STAFFING_BYSHIFT = { # FGI team: # manhours / 8hr shift
            1: {'structure': 16, 'systems': 60, 'declam': 16, 'test': 18},
            2: {'structure': 6, 'systems': 14, 'declam': 4, 'test': 0},
            3: {'structure': 0, 'systems': 0, 'declam': 0, 'test': 0}
        }

FGI_CPJ = { # FGI team: # manhours consumed / 1 BTG completed
        'structure': 6,
        'systems': 3.5,
        'declam': 3,
        'test': 8
    }

CENTERLINES = {
    'A1': None,
    'A2': None,
    'A3': None,
    'A4': None,
    'A5': None,
    'A6': None,
    'A7': None,
    'A8': None,
    'A9': ['C1'],
    'A10': ['C1', 'C2'],
    'BSC1': ['C1', 'C2', 'C3', 'C3.5', 'C4'],
    'BSC2': ['C1', 'C2', 'C3', 'C3.5', 'C4', 'C5'],
    'C1': None,
    'C2': ['C1'],
    'C3': ['C1', 'C2'],
    'C3.5': ['C1', 'C2', 'C3'],
    'C4': ['C1', 'C2', 'C3', 'C3.5'],
    'C5': ['C1', 'C2', 'C3', 'C3.5', 'C4'],
    'CR1': ['CR3', 'CR2'],
    'CR2': ['CR3'],
    'CR3': None,
    'D1': None,
    'D2': None,
    'F1': ['C1', 'C2'],
    'F2': ['C1', 'C2'],
    'L4': None,
    'L5': ['L4'],
    'Spur': None,
    'T1': None,
    'T2': None,
    'T3': None,
    'T4': None
}

FGI_TEAMS = ['structure', 'systems', 'declam', 'test']
BTG_TYPES = ['tot', 'p0', 'p1', 'p2', 'p3', 'engines', 'doors', 'test']

BTG_TYPES_BY_LABOR = { # relationships ONLY, NOT 1-to-1 btg conversion
    'structure': ['tot'],
    'systems': ['p2'],
    'declam': ['p3', 'engines'],
    'test': ['test']
}
FGI_TEAMS_BY_BTG_TYPE = { # relationships ONLY, NOT 1-to-1 btg conversion
    'tot': ['structure', 'systems', 'declam', 'test'],
    'FGI_tot': ['structure'],
    'p2': ['systems'],
    'p3': ['declam'],
    'engines': ['declam'],
    'test': ['test']
}
for type in BTG_TYPES:
    if type not in FGI_TEAMS_BY_BTG_TYPE.keys():
        FGI_TEAMS_BY_BTG_TYPE[type] = None



## 3. Generated Live-State Data
FARO, location, labor, move-time, and paint-schedule inputs are cleaned and exported to `FGI_liveState.xlsx`.


In [61]:

# dataframe builder functions
def merge_APdata(faro_scorecard, p3_milestones, tankClosure):
    ## organize milestone/tankClosure by LN
    tank_lookup = (
        tankClosure[['LINE_NUMBER', 'TankStatus']]
        .drop_duplicates(subset='LINE_NUMBER')
        .rename(columns={'LINE_NUMBER': 'LN'})
    )
    p3_lookup = (
        p3_milestones
        .rename(columns={'P': 'LN'})
        .copy()
    )

    faro_scorecard = faro_scorecard.copy()
    faro_scorecard['LN'] = pd.to_numeric(faro_scorecard['LN'], errors='coerce').astype('Int64')

    tank_lookup['LN'] = pd.to_numeric(tank_lookup['LN'], errors='coerce').astype('Int64')
    p3_lookup['LN'] = pd.to_numeric(p3_lookup['LN'], errors='coerce').astype('Int64')

    ## store LN as str for merge
    faro_scorecard['LN'] = faro_scorecard['LN'].astype(str)
    tank_lookup['LN'] = tank_lookup['LN'].astype(str)
    p3_lookup['LN'] = p3_lookup['LN'].astype(str)

    ## merge all dataframes
    ap_df = (
        faro_scorecard
        .merge(tank_lookup, on='LN', how='left')
        .merge(p3_lookup, on='LN', how='left')
    )

    return ap_df

def build_ap_df(faro_scorecard, p3_milestones, tankClosure):
    ap_data = merge_APdata(faro_scorecard, p3_milestones, tankClosure)

    rows = []

    for _, row in ap_data.iterrows():
        ln = int(float(row['LN']))

        rows.append({
            'LN': ln,
            'FA RO': row['FA RO'],
            'FA RO to B1R': row['FA RO to B1R'],
            'Total Counters': row['Total Counters'],
            'TankStatus': row['TankStatus'],
            'Ceilings': row['Ceilings'],
            'Initial Tests Run': row['Initial Tests Run'],

            # BTG attributes
            'BTG_tot': row['Total BTG'],
            'BTG_p0': row['P0 BTG'],
            'BTG_p1': row['P1 BTG'],
            'BTG_p2': row['P2 BTG'],
            'BTG_p3': row['P3 BTG'],
            'BTG_engines': row['Engines BTG'],
            'BTG_doors': row['Doors BTG'],
            'BTG_test': row['Test BTG'],

            # P3 milestone attributes
            'P3_Engine Hang': row.get('Engine Hang', 0),
            'P3_Flight Controls': row.get('Flight Controls', 0),
            'P3_Gear Swing': row.get('Gear Swing', 0),
            'P3_Medium Pressure Test': row.get('Medium Pressure Test', 0),
            'P3_Oil On': row.get('Oil On', 0),
            'P3_Power On': row.get('Power On', 0),
            'P3_Engine_Type': row.get('Engine_Type', None),
            'P3_Milestone_Completion_Percentage': row.get('Milestone_Completion_Percentage', 0),

            # Shake attributes
            'shakes_complete': row['shakes_completed'],
            'shakes_req': row['shakes_req']
        })

    return pd.DataFrame(rows)

def build_location_df(fa_priority, centerlines):
    rows = []
    seen_locations = set()

    for _, row in fa_priority.iterrows():
        loc = row['Location']
        seen_locations.add(loc)
        centerline_deps = centerlines.get(loc, None)
        

        rows.append({
            'Location': loc,
            'Future State Priority': row['Future State Priority'],
            'Date Online': row['Date Online'],
            'Owner': row['Owner'],
            'tooling_jacking': row['Jacking'] == 'Y',
            'tooling_wings': row['Wings'] == 'Y',
            'tooling_tankClosure': row['Tank Closure'] == 'Y',
            'centerline_dependencies': None if centerline_deps is None else ', '.join(centerline_deps),
            'obstructions': None,
            'notes': None
        })

    return pd.DataFrame(rows)

def build_labor_df(fgi_staffing_byshift, fgi_cpj):
    rows = []

    # FGI staffing by shift
    for shift, teams in fgi_staffing_byshift.items():
        for team, manhours in teams.items():
            rows.append({
                'category': 'FGI_STAFFING_BYSHIFT',
                'shift': shift,
                'team': team,
                'value': manhours
            })

    # FGI CPJ
    for team, cpj in fgi_cpj.items():
        rows.append({
            'category': 'FGI_CPJ',
            'shift': None,
            'team': team,
            'value': cpj
        })

    return pd.DataFrame(rows)



In [ ]:

# build dataframes from FARO scorecare with milestones

def clean_FAstatus(faro_scorecard, tankClosure, p3_milestones):
    ## FARO Scorecard Cleaning
    # only take rows with valid LNs
    faro_scorecard = faro_scorecard[pd.to_numeric(faro_scorecard['LN'], errors='coerce').notna()].copy()
    # remove unnamed columns
    faro_scorecard = faro_scorecard.loc[:, ~faro_scorecard.columns.astype(str).str.contains(r'^Unnamed')]

    # split zone shakes column into two columns: completed/required
    faro_scorecard[['shakes_completed', 'shakes_req']] = faro_scorecard['Zone Shakes'].astype(str).str.split('/', expand=True)
    # only take 
    faro_scorecard['p3_milestones'] = faro_scorecard['P3 Milestones'].astype(str).str.split('/').str[0]

    # set datatypes
    faro_scorecard = (
        faro_scorecard.assign(
            LN=lambda df: pd.to_numeric(df['LN'], errors='coerce').fillna(0).astype(int),
            **{
                'FA RO to B1R': lambda df: pd.to_numeric(df['FA RO to B1R'], errors='coerce'),
                'Total Counters': lambda df: pd.to_numeric(df['Total Counters'], errors='coerce').fillna(0),
                'Total BTG': lambda df: pd.to_numeric(df['Total BTG'], errors='coerce').fillna(0),
                'P0 BTG': lambda df: pd.to_numeric(df['P0 BTG'], errors='coerce').fillna(0),
                'P1 BTG': lambda df: pd.to_numeric(df['P1 BTG'], errors='coerce').fillna(0),
                'P2 BTG': lambda df: pd.to_numeric(df['P2 BTG'], errors='coerce').fillna(0),
                'P3 BTG': lambda df: pd.to_numeric(df['P3 BTG'], errors='coerce').fillna(0),
                'Engines BTG': lambda df: pd.to_numeric(df['Engines BTG'], errors='coerce').fillna(0),
                'Doors BTG': lambda df: pd.to_numeric(df['Doors BTG'], errors='coerce').fillna(0),
                'Test BTG': lambda df: pd.to_numeric(df['Test BTG'], errors='coerce').fillna(0),
                'Tank Closure': lambda df: df['Tank Closure'].map({'Open': 0, 'Closed': 1}).fillna(0).astype(int),
                'Ceilings': lambda df: pd.to_numeric(df['Ceilings'], errors='coerce').fillna(0),
                'Initial Tests Run': lambda df: (
                    df['Initial Tests Run'].astype(str)
                    .str.replace('%', '', regex=False)
                    .replace('nan', 0)
                    .replace('', 0)
                    .astype(float) / 100
                ),
                'shakes_completed': lambda df: pd.to_numeric(df['shakes_completed'], errors='coerce').fillna(0).astype(int),
                'shake_req': lambda df: pd.to_numeric(df['shakes_req'], errors='coerce').fillna(0).astype(int),
                'p3_milestones': lambda df: pd.to_numeric(df['p3_milestones'], errors='coerce').fillna(0).astype(int),
            }
        )
    )

    ## TANK CLOSURE Cleaning
    tankClosure['LINE_NUMBER'] = pd.to_numeric(tankClosure['LINE_NUMBER'], errors='coerce')
    tankClosure['Complete_Jobs'] = pd.to_numeric(tankClosure['Complete_Jobs'], errors='coerce').fillna(0)
    tankClosure['Total_Jobs'] = pd.to_numeric(tankClosure['Total_Jobs'], errors='coerce').fillna(0)
    tankClosure['TankStatus'] = tankClosure['TankStatus'].map({'Open': 0, 'Closed': 1}).fillna(0).astype(int)

    ## P3 Milestone Cleaning
    p3_milestones = p3_milestones.copy()
    
    # Drop rows with NaN in P column
    p3_milestones = p3_milestones.dropna(subset=['P'])
    
    # Extract engine type from milestone name (e.g., "Engine Hang (RR)" -> "RR")
    p3_milestones['Engine_Type'] = p3_milestones['Milestone'].astype(str).str.extract(r'\((.*?)\)')
    
    # Clean milestone names by removing the engine type suffix
    p3_milestones['Milestone'] = p3_milestones['Milestone'].astype(str).str.replace(r'\s*\(.*?\)', '', regex=True)
    
    # Convert data types
    p3_milestones['P'] = pd.to_numeric(p3_milestones['P'], errors='coerce')
    p3_milestones['Completed_Jobs'] = pd.to_numeric(p3_milestones['Completed_Jobs'], errors='coerce').fillna(0)
    p3_milestones['STATUS (1 Complete, 0 Still Open)'] = pd.to_numeric(p3_milestones['STATUS (1 Complete, 0 Still Open)'], errors='coerce').fillna(0)
    
    # Calculate completion percentage by aircraft (P)
    milestone_completion = p3_milestones.groupby('P')['STATUS (1 Complete, 0 Still Open)'].mean()
    
    # Get first engine type per aircraft
    engine_type_lookup = p3_milestones.groupby('P')['Engine_Type'].first()
    
    # Pivot milestones to columns using Completed_Jobs values
    p3_pivoted = (
        p3_milestones.pivot_table(index='P', columns='Milestone', values='Completed_Jobs', aggfunc='sum')
        .fillna(0)
    )
    
    # Reset index to make 'P' a column
    p3_pivoted = p3_pivoted.reset_index()
    
    # Add back Engine_Type using proper mapping
    p3_pivoted['Engine_Type'] = p3_pivoted['P'].map(engine_type_lookup)
    
    # Add completion percentage
    p3_pivoted['Milestone_Completion_Percentage'] = p3_pivoted['P'].map(milestone_completion)
    
    p3_milestones = p3_pivoted

    return faro_scorecard, tankClosure, p3_milestones

def clean_nodeData(nodes, node_adj):
    ## clean nodes
    nodes = (
        nodes
        .drop(columns=[c for c in nodes.columns if str(c).startswith('Unnamed')], errors='ignore')
        .dropna(how='all')
        .assign(
            node_id=lambda df: df['node_id'].astype('string').str.strip(),
            x=lambda df: pd.to_numeric(df['x'], errors='coerce'),
            y=lambda df: pd.to_numeric(df['y'], errors='coerce'),
            type=lambda df: df['type'].astype('string').str.strip(),
            req_centerline=lambda df: df['req_centerline'].astype('string').str.strip()
        )
        .replace({'': pd.NA})
        .dropna(subset=['node_id', 'x', 'y'])
        .reset_index(drop=True)
    )

    ## clean node_adj 
    node_adj = (
        node_adj
        .drop(columns=[c for c in node_adj.columns if str(c).startswith('Unnamed')], errors='ignore')
        .dropna(how='all')
        .assign(
            from_node=lambda df: df['from_node'].astype('string').str.strip(),
            to_node=lambda df: df['to_node'].astype('string').str.strip()
        )
        .replace({'': pd.NA})
        .dropna(subset=['from_node', 'to_node'])
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return nodes, node_adj

def import_data(rootpath, filepaths = {'FAstatus': 'FA_Status_FGI_Handoff.xlsx','FGI_Locations': 'FGI_Locations_wPriority.xlsx','Nodes': 'Nodes.xlsx'}):
    
    ## FA STATUS DATA IMPORT
    path = filepaths['FAstatus']
    # read FARO scorecard as df
    faro_scorecard = pd.read_excel(path, sheet_name='FARO_Scorecard',header=2,engine='openpyxl')
    tankClosure = pd.read_excel(path,sheet_name='Tank_Closure_Detail',engine='openpyxl')
    p3_milestones = pd.read_excel(path, sheet_name='P3 Milestone Detail',engine='openpyxl')

    ## FGI LOCATION DATA IMPORT
    path = filepaths['FGI_Locations']
    fa_priority = pd.read_excel(path, sheet_name='FA Priority', header=1,engine='openpyxl')

    ## NODES DATA IMPORT
    path = filepaths['Nodes']
    nodes = pd.read_excel(path,sheet_name='Node',engine='openpyxl')
    node_adj = pd.read_excel(path, sheet_name='adjacency',engine='openpyxl')

    ## DATA CLEANING
    faro_scorecard, tankClosure, p3_milestones = clean_FAstatus(faro_scorecard, tankClosure, p3_milestones)

    fa_priority = (
        fa_priority
        .drop(columns=[c for c in fa_priority.columns if str(c).startswith('Unnamed')], errors='ignore')
        .dropna(how='all')
        .reset_index(drop=True)
    )

    nodes, node_adj = clean_nodeData(nodes, node_adj)

    return faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj

faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj = import_data(rootpath, filepaths)
ap_df = build_ap_df(faro_scorecard, p3_milestones, tankClosure)

location_df = build_location_df(
    fa_priority=fa_priority,
    centerlines=CENTERLINES
)

labor_df = build_labor_df(
    fgi_staffing_byshift=FGI_STAFFING_BYSHIFT,
    fgi_cpj=FGI_CPJ
)


staffing_df = labor_df[labor_df['category'] == 'FGI_STAFFING_BYSHIFT'].copy()
staffing_df['shift'] = staffing_df['shift'].astype(int)
staffing_df['value'] = pd.to_numeric(staffing_df['value'])

FGI_STAFFING_BYSHIFT = {
    shift: skill.set_index('team')['value'].to_dict()
    for shift, skill in staffing_df.groupby('shift', sort=True)
}

FGI_CPJ = (
    labor_df[labor_df['category'] == 'FGI_CPJ']
    .assign(value=lambda df: pd.to_numeric(df['value']))
    .set_index('team')['value']
    .to_dict()
)


move_times_df = pd.read_excel(filepaths['Move Times'], index_col=0)
move_times_df.index = move_times_df.index.astype(str)
move_times_df.columns = move_times_df.columns.astype(str)

paint_schedule_df = pd.read_excel(
    filepaths['Paint Schedule'],
    sheet_name='Historical',
    header=2,
    engine='openpyxl'
)[['Date', 'BSC1', 'BSC2']].copy()
paint_schedule_df['Date'] = pd.to_datetime(paint_schedule_df['Date'], errors='coerce').dt.normalize()
paint_schedule_df = paint_schedule_df[paint_schedule_df['Date'].notna()].reset_index(drop=True)



In [63]:
# export generated live-state workbook
with pd.ExcelWriter(LIVE_STATE_FILE, engine='openpyxl') as writer:
    ap_df.to_excel(writer, sheet_name='ap_data', index=False)
    location_df.to_excel(writer, sheet_name='location_data', index=False)
    labor_df.to_excel(writer, sheet_name='labor_data', index=False)
    move_times_df.to_excel(writer, sheet_name='move_times')
    paint_schedule_df.to_excel(writer, sheet_name='paint_schedule', index=False)

print(LIVE_STATE_FILE)


c:\Users\dcube\BSC-FGI_Scheduler\FGI_liveState.xlsx


In [64]:

# DEBUG: Simpler check - what are the unique LN and P values?
print("Unique LN values in FARO scorecard:")
print(sorted(faro_scorecard['LN'].unique()))

print("\nUnique P values in P3 milestones:")
print(sorted(p3_milestones['P'].unique()))

print("\n'P' column in FARO scorecard?")
print('P' in faro_scorecard.columns)

if 'P' in faro_scorecard.columns:
    print("\nFARO scorecard LN and P values:")
    print(faro_scorecard[['LN', 'P']].drop_duplicates().sort_values('LN').head(20))



Unique LN values in FARO scorecard:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93]

Unique P values in P3 milestones:
[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 53.0, 54.0, 55.0, 56.0, 57.0, 58.0, 59.0, 60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 67.0, 68.0, 69.0, 70.0, 71.0, 72.0, 73.0, 74.0, 75.0, 76.0, 77.0, 78.0, 79.0, 80.0, 81.0, 82.0, 83.0, 84.0, 85.0, 86.0, 87.0, 88.0, 89.0, 90.0, 91.0, 92.0]

'P' column in FARO s

In [65]:

# Quick merge test
print("Before merge - p3_milestones P values:", p3_milestones['P'].unique())
print("FARO LN sample:", faro_scorecard['LN'].head(10).tolist())

# Test the merge directly
test_merge = faro_scorecard[['LN']].merge(
    p3_milestones.rename(columns={'P': 'LN'}),
    on='LN',
    how='left'
)
print(f"\nMerge result - non-null rows: {test_merge.notna().any(axis=1).sum()} out of {len(test_merge)}")
print("\nMerge result sample:")
print(test_merge.head(10))



Before merge - p3_milestones P values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.
 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34. 35. 36.
 37. 38. 39. 40. 41. 42. 43. 44. 45. 46. 47. 48. 49. 50. 51. 52. 53. 54.
 55. 56. 57. 58. 59. 60. 61. 62. 63. 64. 65. 66. 67. 68. 69. 70. 71. 72.
 73. 74. 75. 76. 77. 78. 79. 80. 81. 82. 83. 84. 85. 86. 87. 88. 89. 90.
 91. 92.]
FARO LN sample: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Merge result - non-null rows: 93 out of 93

Merge result sample:
   LN  Engine Hang  Flight Controls  Gear Swing  Medium Pressure Test  Oil On  \
0   1          2.0              6.0         1.0                   2.0     1.0   
1   2          2.0              6.0         1.0                   2.0     1.0   
2   3          2.0              6.0         1.0                   2.0     1.0   
3   4          2.0              6.0         1.0                   2.0     1.0   
4   5          2.0              6.0         1.0                   2

In [66]:

# Deep dive into the merge issue
ap_data_test = merge_APdata(faro_scorecard, p3_milestones, tankClosure)

print("p3_milestones columns and first row:")
print(p3_milestones.columns.tolist())
print(f"\np3_milestones shape: {p3_milestones.shape}")
print(f"p3_milestones 'P' column (first 5): {p3_milestones['P'].head().tolist()}")

print("\n\nap_data_test first row P3 milestone values:")
print(f"LN: {ap_data_test.iloc[0]['LN']}")
print(f"Engine Hang: {ap_data_test.iloc[0].get('Engine Hang', 'KEY NOT FOUND')}")
print(f"Flight Controls: {ap_data_test.iloc[0].get('Flight Controls', 'KEY NOT FOUND')}")

print("\n\nRows where Engine Hang is NOT NaN:")
en_hang_col = ap_data_test['Engine Hang']
non_null = en_hang_col.notna()
print(f"Count: {non_null.sum()}")
if non_null.sum() > 0:
    print("Sample rows with data:")
    print(ap_data_test[non_null][['LN', 'Engine Hang', 'Flight Controls']].head())

print("\n\nChecking LN values:")
print(f"FARO LN range: {ap_data_test['LN'].min()} to {ap_data_test['LN'].max()}")
print(f"First 10 LN values: {ap_data_test['LN'].head(10).tolist()}")



p3_milestones columns and first row:
['P', 'Engine Hang', 'Flight Controls', 'Gear Swing', 'Medium Pressure Test', 'Oil On', 'Power On', 'Engine_Type', 'Milestone_Completion_Percentage']

p3_milestones shape: (92, 9)
p3_milestones 'P' column (first 5): [1.0, 2.0, 3.0, 4.0, 5.0]


ap_data_test first row P3 milestone values:
LN: 1
Engine Hang: 2.0
Flight Controls: 6.0


Rows where Engine Hang is NOT NaN:
Count: 92
Sample rows with data:
  LN  Engine Hang  Flight Controls
0  1          2.0              6.0
1  2          2.0              6.0
2  3          2.0              6.0
3  4          2.0              6.0
4  5          2.0              6.0


Checking LN values:
FARO LN range: 1 to 93
First 10 LN values: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']


In [67]:

# Re-run import and build with the fixed merge function
faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj = import_data(rootpath, filepaths)
ap_df = build_ap_df(faro_scorecard, p3_milestones, tankClosure)

# Verify P3 data is now populated
p3_cols = [col for col in ap_df.columns if col.startswith('P3_')]
print("P3 columns in ap_df:")
print(p3_cols)
print("\nFirst row P3 values:")
print(ap_df.iloc[0][p3_cols])
print("\nRow with non-zero P3_Engine Hang:")
non_zero = ap_df[ap_df['P3_Engine Hang'] > 0].head()
print(non_zero[['LN'] + p3_cols])



P3 columns in ap_df:
['P3_Engine Hang', 'P3_Flight Controls', 'P3_Gear Swing', 'P3_Medium Pressure Test', 'P3_Oil On', 'P3_Power On', 'P3_Engine_Type', 'P3_Milestone_Completion_Percentage']

First row P3 values:
P3_Engine Hang                        2.0
P3_Flight Controls                    6.0
P3_Gear Swing                         1.0
P3_Medium Pressure Test               2.0
P3_Oil On                             1.0
P3_Power On                           1.0
P3_Engine_Type                         RR
P3_Milestone_Completion_Percentage    1.0
Name: 0, dtype: object

Row with non-zero P3_Engine Hang:
   LN  P3_Engine Hang  P3_Flight Controls  P3_Gear Swing  \
0   1             2.0                 6.0            1.0   
1   2             2.0                 6.0            1.0   
2   3             2.0                 6.0            1.0   
3   4             2.0                 6.0            1.0   
4   5             2.0                 6.0            1.0   

   P3_Medium Pressure Test  P3_Oi